In [2]:
from pyspark.sql import functions as F

# 1. Silver transform
BRONZE = "user_bronze"
SILVER = "user_silver"

df_user = spark.table(BRONZE)

base_cols = [
"user_id","name","review_count","yelping_since","average_stars",
"useful","funny","cool","fans","friends","elite",
"compliment_cool","compliment_cute","compliment_funny","compliment_hot",
"compliment_list","compliment_more","compliment_note","compliment_photos",
"compliment_plain","compliment_profile","compliment_writer",
"_ingest_ts","_source_file","_batch_id"
]

df_user_silver = df_user.select(*base_cols)

StatementMeta(, 676e2a74-abdd-4fb0-857e-bdc8a633c44e, 4, Finished, Available, Finished, False)

In [3]:
df_user_silver = (
    df_user_silver
    .withColumn("review_count", F.col("review_count").cast("int"))
    .withColumn("average_stars", F.col("average_stars").cast("double"))

    # vote 
    .withColumn("useful", F.coalesce(F.col("useful"), F.lit(0)).cast("int"))
    .withColumn("funny", F.coalesce(F.col("funny"), F.lit(0)).cast("int"))
    .withColumn("cool", F.coalesce(F.col("cool"), F.lit(0)).cast("int"))
    .withColumn("fans", F.coalesce(F.col("fans"), F.lit(0)).cast("int"))

    # compliment 
    .withColumn("compliment_cool", F.coalesce(F.col("compliment_cool"), F.lit(0)).cast("int"))
    .withColumn("compliment_cute", F.coalesce(F.col("compliment_cute"), F.lit(0)).cast("int"))
    .withColumn("compliment_funny", F.coalesce(F.col("compliment_funny"), F.lit(0)).cast("int"))
    .withColumn("compliment_hot", F.coalesce(F.col("compliment_hot"), F.lit(0)).cast("int"))
    .withColumn("compliment_list", F.coalesce(F.col("compliment_list"), F.lit(0)).cast("int"))
    .withColumn("compliment_more", F.coalesce(F.col("compliment_more"), F.lit(0)).cast("int"))
    .withColumn("compliment_note", F.coalesce(F.col("compliment_note"), F.lit(0)).cast("int"))
    .withColumn("compliment_photos", F.coalesce(F.col("compliment_photos"), F.lit(0)).cast("int"))
    .withColumn("compliment_plain", F.coalesce(F.col("compliment_plain"), F.lit(0)).cast("int"))
    .withColumn("compliment_profile", F.coalesce(F.col("compliment_profile"), F.lit(0)).cast("int"))
    .withColumn("compliment_writer", F.coalesce(F.col("compliment_writer"), F.lit(0)).cast("int"))

    .withColumn("name", F.trim(F.col("name")))

    .filter(F.col("user_id").isNotNull())
    .filter(F.col("average_stars").isNotNull())
    .filter((F.col("average_stars") >= 1) & (F.col("average_stars") <= 5))
    .filter(F.col("review_count") >= 0)

    .dropDuplicates(["user_id"])
)


StatementMeta(, 676e2a74-abdd-4fb0-857e-bdc8a633c44e, 5, Finished, Available, Finished, False)

In [4]:
(df_user_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)
)

StatementMeta(, 676e2a74-abdd-4fb0-857e-bdc8a633c44e, 6, Finished, Available, Finished, False)

In [5]:
df_user_silver.printSchema()

StatementMeta(, 676e2a74-abdd-4fb0-857e-bdc8a633c44e, 7, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: integer (nullable = true)
 |-- yelping_since: string (nullable = true)
 |-- average_stars: double (nullable = true)
 |-- useful: integer (nullable = false)
 |-- funny: integer (nullable = false)
 |-- cool: integer (nullable = false)
 |-- fans: integer (nullable = false)
 |-- friends: string (nullable = true)
 |-- elite: string (nullable = true)
 |-- compliment_cool: integer (nullable = false)
 |-- compliment_cute: integer (nullable = false)
 |-- compliment_funny: integer (nullable = false)
 |-- compliment_hot: integer (nullable = false)
 |-- compliment_list: integer (nullable = false)
 |-- compliment_more: integer (nullable = false)
 |-- compliment_note: integer (nullable = false)
 |-- compliment_photos: integer (nullable = false)
 |-- compliment_plain: integer (nullable = false)
 |-- compliment_profile: integer (nullable = false)
 |-- compliment_writer: integer (nullable = false)
 |-- _i

In [6]:
print("user_silver counts: ", spark.table("user_silver").count())

StatementMeta(, 676e2a74-abdd-4fb0-857e-bdc8a633c44e, 8, Finished, Available, Finished, False)

user_silver counts:  1987897


In [ ]:
mssparkutils.notebook.exit("SUCCESS")

In [7]:
#2. Data Quality Check 
df_user_silver_DQ = spark.table("user_silver")

print("user rows = ", df_user_silver.count())

dup = (
    df_user_silver.groupBy("user_id")
    .count()
    .filter(F.col("count") > 1 )
    .count()
)

print("duplicate user_id = ", dup)

df_user_silver_DQ.select(
    F.sum(F.col("user_id").isNull().cast("int")).alias("null_user_id"),
    F.sum(F.col("review_count").isNull().cast("int").alias("null_review_count")),
    F.sum(F.col("average_stars").isNull().cast("int").alias("null_average_stars"))
).show()

StatementMeta(, 676e2a74-abdd-4fb0-857e-bdc8a633c44e, 9, Finished, Available, Finished, False)

user rows =  1987897
duplicate user_id =  0
+------------+-------------------------------------------------------------+---------------------------------------------------------------+
|null_user_id|sum(CAST((review_count IS NULL) AS INT) AS null_review_count)|sum(CAST((average_stars IS NULL) AS INT) AS null_average_stars)|
+------------+-------------------------------------------------------------+---------------------------------------------------------------+
|           0|                                                            0|                                                              0|
+------------+-------------------------------------------------------------+---------------------------------------------------------------+

